In [ ]:
pip install gdown

In [ ]:
!pip install ultralytics

In [ ]:
import gdown
import zipfile
import logging
import ultralytics
from ultralytics import YOLO
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from PIL import Image
import os
import cv2

In [ ]:
file_id = '1RwB_X8SxQnqoVT7IiTQD9GsX9-FRd8Ib'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'dataset.zip'
gdown.download(url, output, quiet=False)

In [ ]:
with zipfile.ZipFile('dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('dataset_folder')

In [ ]:
import os

base_path = '/content/dataset_folder'


folders = [
    'resultImages/train', 'resultImages/val', 'resultImages/test',
    'labels/train', 'labels/val', 'labels/test'
]

for folder in folders:
    path = os.path.join(base_path, folder)
    os.makedirs(path, exist_ok=True)
    print(f"Klasör oluşturuldu (veya zaten var): {path}")

In [ ]:
import shutil
import os

base_path = '/content/dataset_folder'

images_dir = os.path.join(base_path, 'Images')
annotations_dir = os.path.join(base_path, 'Annotations')

train_list_path = os.path.join(base_path, 'ImageSets/train.txt')
val_list_path = os.path.join(base_path, 'ImageSets/val.txt')
test_list_path = os.path.join(base_path, 'ImageSets/test.txt')

def move_files(list_path, split):
    with open(list_path, 'r') as f:
        lines = f.read().strip().splitlines()

    for name in lines:
        img_png = os.path.join(images_dir, f'{name}.png')
        label_src = os.path.join(annotations_dir, f'{name}.txt')

        img_dst_dir = os.path.join(base_path, f'resultImages/{split}')
        label_dst_dir = os.path.join(base_path, f'labels/{split}')

        if os.path.exists(img_png):
            shutil.copy(img_png, img_dst_dir)
            print(f"{img_png} -> {img_dst_dir}")
        else:
            print(f"Uyarı: Görsel bulunamadı {name}")

        if os.path.exists(label_src):
            shutil.copy(label_src, label_dst_dir)
            print(f"{label_src} -> {label_dst_dir}")
        else:
            print(f"Uyarı: Label bulunamadı {name}")

move_files(train_list_path, 'train')
move_files(val_list_path, 'val')
move_files(test_list_path, 'test')

In [ ]:
from PIL import Image

splits = ['train', 'val', 'test']

for split in splits:
    label_dir = os.path.join(base_path, f'labels/{split}')
    image_dir = os.path.join(base_path, f'resultImages/{split}')

    for filename in os.listdir(label_dir):
        if filename.endswith('.txt'):
            label_path = os.path.join(label_dir, filename)

            img_name_jpg = filename.replace('.txt', '.jpg')
            img_name_png = filename.replace('.txt', '.png')

            if os.path.exists(os.path.join(image_dir, img_name_jpg)):
                img_path = os.path.join(image_dir, img_name_jpg)
            elif os.path.exists(os.path.join(image_dir, img_name_png)):
                img_path = os.path.join(image_dir, img_name_png)
            else:
                print(f"Görsel bulunamadı: {filename}")
                continue

            with Image.open(img_path) as img:
                image_width, image_height = img.size

            with open(label_path, 'r') as f:
                lines = f.read().strip().splitlines()

            new_lines = []
            for line in lines:
                parts = line.strip().split()
                if len(parts) != 5:
                    print(f"Hatalı format {filename}: {line}")
                    continue

                x_min = float(parts[0])
                y_min = float(parts[1])
                x_max = float(parts[2])
                y_max = float(parts[3])
                class_id = int(parts[4])

                class_id = 0

                x_center = (x_min + x_max) / 2 / image_width
                y_center = (y_min + y_max) / 2 / image_height
                width = (x_max - x_min) / image_width
                height = (y_max - y_min) / image_height

                new_line = f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
                new_lines.append(new_line)

            with open(label_path, 'w') as f:
                f.write('\n'.join(new_lines))

            print(f"Dönüştürüldü: {filename} ({split})")

In [ ]:
yaml_content = """
path: /content/dataset_folder
train: images/train
val: images/val
test: _backup_test_set/images/test

nc: 1
names: ['car']
"""

with open('/content/dataset_folder/dataset.yaml', 'w') as f:
    f.write(yaml_content)


In [ ]:
import shutil
import os

folder_paths = ['/content/dataset_folder/Annotations','/content/dataset_folder/ImageSets','/content/dataset_folder/Images']

for fp in folder_paths:
    if os.path.exists(fp):
        shutil.rmtree(fp)
        print(f"Klasör silindi: {fp}")
    else:
        print(f"Klasör zaten yok: {fp}")

In [ ]:
backup_dir = '/content/dataset_folder/_backup_test_set'
backup_labels = os.path.join(backup_dir, 'labels')
backup_images = os.path.join(backup_dir, 'images')

os.makedirs(backup_labels, exist_ok=True)
os.makedirs(backup_images, exist_ok=True)

shutil.move('/content/dataset_folder/labels/test', backup_labels)
shutil.move('/content/dataset_folder/resultImages/test', backup_images)


In [ ]:
old_dir = '/content/dataset_folder/resultImages'
new_dir = '/content/dataset_folder/images'

os.rename(old_dir, new_dir)
print(f"Klasör yeniden adlandırıldı: {old_dir} -> {new_dir}")

In [ ]:
def resultInf(results):
  p = results.box.p[0]
  r = results.box.r[0]
  map50 = results.box.map50
  map95 = results.box.map

  print('Results Of Test Set')
  print(f"Precision: {p:.3f}")
  print(f"Recall: {r:.3f}")
  print(f"F1-score: {2 * (p * r) / (p + r):.3f}")
  print(f"mAP@0.5: {map50:.3f}")
  print(f"mAP@0.5:0.95: {map95:.3f}")
  print(results.confusion_matrix.matrix)

In [ ]:
def evaluateCounting(predicted_counts, ground_truth_counts):
    mse = mean_squared_error(ground_truth_counts, predicted_counts)
    exact_matches = sum([p == g for p, g in zip(predicted_counts, ground_truth_counts)])
    exact_match_accuracy = 100 * exact_matches / len(ground_truth_counts)

    print(f"Exact Match Accuracy: {exact_match_accuracy:.2f}%")
    print(f"Mean Squared Error: {mse:.4f}")

def get_ground_truth_counts(label_dir, image_names):
    counts = []
    for name in image_names:
        base_name = name.replace('.jpg', '').replace('.png', '')
        label_path = os.path.join(label_dir, base_name + '.txt')

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                count = len(f.readlines())
        else:
            print(f"⚠️ Etiket bulunamadı: {label_path}")
            count = 0

        counts.append(count)
    return counts

In [ ]:
def draw_boxes_comparison(prediction_obj, label_path, image_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape

    img_pred = img.copy()
    img_gt = img.copy()

    if prediction_obj.boxes is not None:
        for box in prediction_obj.boxes:
            b = box.xyxy[0].cpu().numpy().astype(int)
            cv2.rectangle(img_pred, (b[0], b[1]), (b[2], b[3]), (255, 0, 0), 2)

    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            lines = f.readlines()
            for line in lines:
                parts = list(map(float, line.strip().split()))
                cls, cx, cy, bw, bh = parts
                x1 = int((cx - bw / 2) * w)
                y1 = int((cy - bh / 2) * h)
                x2 = int((cx + bw / 2) * w)
                y2 = int((cy + bh / 2) * h)
                cv2.rectangle(img_gt, (x1, y1), (x2, y2), (0, 255, 0), 2)

    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(img_gt)
    plt.title("Ground Truth")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(img_pred)
    plt.title("Prediction")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
def createModelTrainAndTest(optimizer='Adam', learningRate=1e-4, batchSize=16, freezeOrNot=False, freezeValue=0,epochVal = 20,imgSize = 640):
    logging.getLogger("ultralytics").setLevel(logging.WARNING)  # Konsol çıktısını azalt

    model = YOLO('yolov8n.pt')

    runName = f"YOLO_lr{learningRate}_b{batchSize}_f{freezeValue}_e{epochVal}"

    train_kwargs = {
        'data': '/content/dataset_folder/dataset.yaml',
        'epochs': epochVal,
        'imgsz': imgSize,
        'batch': batchSize,
        'optimizer': optimizer,
        'lr0': learningRate,
        'verbose': False,
        'project': 'runs/detect',
        'name': runName
    }

    if freezeOrNot:
        train_kwargs['freeze'] = freezeValue

    model.train(**train_kwargs)

    val_metrics = model.val(data='dataset_folder/dataset.yaml', split='test')
    resultInf(val_metrics)

    predictions = model.predict(source='/content/dataset_folder/_backup_test_set/images/test', save=False)
    predicted_counts = [len(p.boxes.data) for p in predictions]

    for pred in predictions[:5]:
        img_path = pred.path
        label_path = img_path.replace('images', 'labels').replace('.png', '.txt')
        draw_boxes_comparison(pred, label_path, img_path)

    image_names = [os.path.basename(p.path) for p in predictions]
    gt_counts = get_ground_truth_counts('/content/dataset_folder/_backup_test_set/labels/test', image_names)
    print('predicteed car count in showing images', sum(predicted_counts[:5]))
    print('ground truth car count in showing images', sum(gt_counts[:5]))
    print('total predicted car count: ',sum(predicted_counts))
    print('total ground truth car count: ',sum(gt_counts))

    evaluateCounting(predicted_counts, gt_counts)

    result_image_path = f"runs/detect/{runName}/results.png"
    if os.path.exists(result_image_path):
        img = Image.open(result_image_path)
        plt.figure(figsize=(16, 12))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"Loss & Metrics - {runName}")
        plt.show()
    else:
        print("results.png not found.")


# BATCH SIZE EXPERIMENTS

# Optimizer = Adam , LearningRate = 1e-4 , BatchSize 16


In [ ]:
createModelTrainAndTest(optimizer='Adam',learningRate=1e-4,batchSize = 16, freezeOrNot= False, freezeValue= 0,epochVal=25)

In [ ]:
createModelTrainAndTest(optimizer='Adam', learningRate=1e-4, batchSize=16, freezeOrNot=True, freezeValue=5,epochVal=25)

In [ ]:
createModelTrainAndTest(optimizer='Adam', learningRate=1e-4, batchSize=16, freezeOrNot=True, freezeValue=10,epochVal=25)

In [ ]:
createModelTrainAndTest(optimizer='Adam', learningRate=1e-4, batchSize=16, freezeOrNot=True, freezeValue=21,epochVal=25)

# Optimizer = Adam , LearningRate = 1e-4 , BatchSize 64

In [ ]:
createModelTrainAndTest(optimizer='Adam',learningRate=1e-4,batchSize = 64, freezeOrNot= False, freezeValue= 0,epochVal=25)

In [ ]:
createModelTrainAndTest(optimizer='Adam', learningRate=1e-4, batchSize=64, freezeOrNot=True, freezeValue=5,epochVal=25)

In [ ]:
createModelTrainAndTest(optimizer='Adam', learningRate=1e-4, batchSize=64, freezeOrNot=True, freezeValue=10,epochVal=25)

In [ ]:
createModelTrainAndTest(optimizer='Adam', learningRate=1e-4, batchSize=64, freezeOrNot=True, freezeValue=21,epochVal=25)

| Model | Batch Size | Freeze | Optimizer | Learning Rate |  MSE  | Accuracy (%) |
|-------|------------|--------|-----------|----------------|-------|---------------|
|   1   |     16     |   0    |   Adam    |       1e-4         |  22.5   |    25    |
|   2   |     16     |   5    |   Adam    |       1e-4         |  53.5   |    27.5  |
|   3   |     16     |   10   |   Adam    |      1e-4          |  57   |    18.5    |
|   4   |     16     |   21   |   Adam    |       1e-4         |  985 |    2   |
|   5   |     64     |   0    |   Adam    |       1e-4         |  27.9   |    26    |
|   6   |     64     |   5    |   Adam    |       1e-4         |  54   |    20  |
|   7   |     64     |   10   |   Adam    |      1e-4          |   55.5   |    14.5    |
|   8   |     64     |   21   |   Adam    |       1e-4         |  1085 |    1.5   |

The extended results further emphasize the strong interaction between batch size and the number of frozen layers (Freeze) on the model’s performance, particularly in the context of a low learning rate (1e-4) and a limited dataset of only 1000 images. When all layers are trainable (Freeze = 0), the model consistently produces the best results, with Model 1 (batch size = 16) achieving the lowest MSE (22.5) and a solid accuracy of 25%. Interestingly, Model 2 (Freeze = 5) slightly outperforms it in accuracy (27.5%) while exhibiting a higher MSE, suggesting that a moderate amount of layer freezing may occasionally aid stability, though at the cost of increased variance in count predictions. As the number of frozen layers increases to 10 and especially 21, performance declines rapidly. Model 4 and 8, both trained with Freeze = 21, show catastrophic degradation with MSE values close to or exceeding 1000 and accuracy dropping to nearly zero, confirming that freezing too much of the network renders it nearly incapable of adapting to the new task.

This performance drop is expected, as frozen layers retain the weights learned from a pretraining domain (e.g., ImageNet) that may not match the structure and scale of the drone-based car counting task. When only the detection head is trainable, the model is forced to interpret features that may not be discriminative for this specific dataset, leading to poor generalization and inaccurate predictions. The inability to adjust low-level and mid-level feature extractors means that the model cannot learn dataset-specific characteristics, particularly when the task involves subtle visual cues like counting overlapping or partially occluded cars.

The batch size analysis reveals a similar pattern. Across all freeze levels, models trained with a batch size of 16 consistently outperform those with a batch size of 64. While the difference in MSE and accuracy is not always drastic at lower freeze levels, it becomes more pronounced as more layers are frozen. This supports the hypothesis that smaller batch sizes allow for more frequent and responsive updates, which is especially beneficial in low-data scenarios where every gradient update carries significant learning signal. Larger batch sizes, on the other hand, tend to average out these signals, potentially slowing convergence and leading to poorer local minima — particularly when combined with limited trainability due to freezing.

In conclusion, the results reinforce the importance of minimizing layer freezing and favoring smaller batch sizes when training YOLOv8 models on small datasets under constrained learning conditions. The best-performing configurations are those that allow the model to fully adapt its internal representations to the specific structure and variability of the dataset. These findings align with transfer learning principles: unless the pretrained features are highly aligned with the new task, restricting the model’s capacity to adapt can be detrimental to performance.

# LEARNING RATE EXPERIMENTS

# Optimizer = Adam , LearningRate = 1e-3 , BatchSize 16

In [ ]:
createModelTrainAndTest(optimizer='Adam',learningRate=1e-3,batchSize = 16, freezeOrNot= False, freezeValue= 0,epochVal=25)

In [ ]:
createModelTrainAndTest(optimizer='Adam', learningRate=1e-3, batchSize=16, freezeOrNot=True, freezeValue=5,epochVal=25)

In [ ]:
createModelTrainAndTest(optimizer='Adam', learningRate=1e-3, batchSize=16, freezeOrNot=True, freezeValue=10,epochVal=25)

In [ ]:
createModelTrainAndTest(optimizer='Adam', learningRate=1e-3, batchSize=16, freezeOrNot=True, freezeValue=21,epochVal = 25)

# Optimizer = Adam , LearningRate = 1e-5 , BatchSize 16

In [ ]:
createModelTrainAndTest(optimizer='Adam',learningRate=1e-5,batchSize = 16, freezeOrNot= False, freezeValue= 0,epochVal=25)

In [ ]:
createModelTrainAndTest(optimizer='Adam', learningRate=1e-5, batchSize=16, freezeOrNot=True, freezeValue=5,epochVal=25)

In [ ]:
createModelTrainAndTest(optimizer='Adam', learningRate=1e-5, batchSize=16, freezeOrNot=True, freezeValue=10,epochVal=25)

In [ ]:
createModelTrainAndTest(optimizer='Adam', learningRate=1e-5, batchSize=16, freezeOrNot=True, freezeValue=21,epochVal=25)

| Model | Batch Size | Freeze | Optimizer | Learning Rate |  MSE  | Accuracy (%) |
|-------|------------|--------|-----------|----------------|-------|---------------|
|   1   |     16     |   0    |   Adam    |      1e-3          |  8.36   |    31.5         |
|   2   |     16     |   5    |   Adam    |       1e-3         |  10.35   |    36       |
|   3   |     16     |   10   |   Adam    |       1e-3         |  11.76   |    33         |
|   4   |     16     |   21   |   Adam    |       1e-3         | 207  |    5        |
|   5   |     16     |   0    |   Adam    |       1e-4         |  22.5   |    25    |
|   6   |     16     |   5    |   Adam    |       1e-4         |  53.5   |    27.5  |
|   7   |     16     |   10   |   Adam    |      1e-4          |  57   |    18.5    |
|   8   |     16     |   21   |   Adam    |       1e-4         |  985 |    2   |
|   9   |     16     |   0    |   Adam    |       1e-5         |  416  |    14.5        |
|  10   |     16     |   5    |   Adam    |       1e-5         |  843.7  |    2.5          |
|  11   |     16     |   10   |   Adam    |           1e-5     |  849  |    0.5          |
|  12   |     16     |   21   |   Adam    |               1e-5 | 4724  |    0          |

The table demonstrates a clear relationship between learning rate, freeze level, and model performance in object counting tasks using YOLOv8. Among the tested learning rates, 1e-3 consistently yields the best results, with Model 2 (Freeze = 5) achieving the highest accuracy (36%) and Model 1 (Freeze = 0) achieving the lowest MSE (8.36). These findings suggest that a moderately high learning rate allows the model to effectively update weights and adapt to the dataset, especially when more layers are trainable. In contrast, as the learning rate decreases to 1e-4, model performance moderately degrades — likely due to slower convergence — and further deteriorates at 1e-5, where the model fails to learn meaningful representations, resulting in extremely high MSE and near-zero accuracy. Additionally, freezing too many layers (Freeze = 21) leads to performance collapse across all learning rates, emphasizing the importance of full or partial fine-tuning when adapting YOLOv8 to new domains. Overall, a learning rate of 1e-3 combined with minimal or moderate freezing provides the most effective configuration under the current dataset constraints.

If the number of training epochs is increased, models with smaller learning rates (e.g., 1e-4 or 1e-5) are expected to perform better over time, as they allow for more gradual and stable convergence. Unlike high learning rates, which can lead to faster initial learning but risk overshooting minima or causing instability, smaller learning rates refine the model weights more precisely, especially in later stages of training. With sufficient epochs, this slow but steady optimization often results in better generalization and lower error on unseen data.

# EPOCH EXPERIMENT

In [ ]:
createModelTrainAndTest(optimizer='Adam',learningRate=1e-3,batchSize = 16, freezeOrNot= False, freezeValue= 0,epochVal= 50)

In [ ]:
createModelTrainAndTest(optimizer='Adam',learningRate=1e-4,batchSize = 16, freezeOrNot= False, freezeValue= 0,epochVal= 50)

In [ ]:
createModelTrainAndTest(optimizer='Adam',learningRate=1e-5,batchSize = 16, freezeOrNot= False, freezeValue= 0,epochVal= 50)

| Model | Batch Size | Freeze | Optimizer | Learning Rate | Epoch |  MSE  | Accuracy (%) |
|-------|------------|--------|-----------|----------------|--------|-------|---------------|
|   1   |     16     |   0    |   Adam    |      1e-3       |  25    |  8.36   |    31.5         |
|   2   |     16     |   0    |   Adam    |      1e-3       |  50    | 4.2  |    42        |
|   3   |     16     |   0   |   Adam    |      1e-4       |  25   | 22.5  |    25        |
|   4   |     16     |   0   |   Adam    |      1e-4       |  50    | 12    |    31.5         |
|   5   |     16     |   0    |   Adam    |      1e-5      |  25    |  416  |    14.5        |
|   6   |     16     |   0    |   Adam    |      1e-5       |  50    |181.75     |    17.5        |

This table clearly illustrates how the learning rate interacts with training duration to influence model performance. At a relatively high learning rate of 1e-3, the model quickly converges to strong results—achieving the best overall performance with Model 2 (50 epochs), which attained the lowest MSE (4.2) and highest accuracy (42%). Even with fewer epochs (Model 1), this learning rate delivers competitive performance. However, at lower learning rates like 1e-4, the model requires more epochs to approach similar results, as seen in Model 4 (MSE 12, Accuracy 31.5%). This confirms that low learning rates demand longer training times to sufficiently minimize error. At the extreme, 1e-5 fails to deliver acceptable results even after 50 epochs, with MSE still above 180 and accuracy below 18%, indicating the updates are too small for effective learning. Overall, the results highlight that 1e-3 is the most suitable learning rate for this dataset and task, particularly when training time is limited.

# OPTIMIZER EXPERIMENTS

# Optimizer = SGD , LearningRate = 1e-3 , BatchSize 16

In [ ]:
createModelTrainAndTest(optimizer='SGD',learningRate=1e-3,batchSize = 16, freezeOrNot= False, freezeValue= 0,epochVal= 25)

In [ ]:
createModelTrainAndTest(optimizer='SGD',learningRate=1e-3,batchSize = 16, freezeOrNot= True, freezeValue= 5,epochVal= 25)

In [ ]:
createModelTrainAndTest(optimizer='SGD',learningRate=1e-3,batchSize = 16, freezeOrNot= True, freezeValue= 10,epochVal= 25)

In [ ]:
createModelTrainAndTest(optimizer='SGD',learningRate=1e-3,batchSize = 16, freezeOrNot= True, freezeValue= 21,epochVal= 25)

# Optimizer = AdamW , LearningRate = 1e-3 , BatchSize 16

In [ ]:
createModelTrainAndTest(optimizer='AdamW',learningRate=1e-3,batchSize = 16, freezeOrNot= False, freezeValue= 0,epochVal= 25)

In [ ]:
createModelTrainAndTest(optimizer='AdamW',learningRate=1e-3,batchSize = 16, freezeOrNot= True, freezeValue= 5,epochVal= 25)

In [ ]:
createModelTrainAndTest(optimizer='AdamW',learningRate=1e-3,batchSize = 16, freezeOrNot= True, freezeValue= 10,epochVal= 25)

In [ ]:
createModelTrainAndTest(optimizer='AdamW',learningRate=1e-3,batchSize = 16, freezeOrNot= True, freezeValue= 21,epochVal= 25)

| Model | Batch Size | Freeze | Optimizer | Learning Rate |  MSE  | Accuracy (%) |
|-------|------------|--------|-----------|----------------|-------|---------------|
|   1   |     16     |   0    |   Adam    |      1e-3          |  8.36   |    31.5         |
|   2   |     16     |   5    |   Adam    |       1e-3         |  10.35   |    36       |
|   3   |     16     |   10   |   Adam    |       1e-3         |  11.76   |    33         |
|   4   |     16     |   21   |   Adam    |       1e-3         | 207  |  5|
|   5   |     16     |   0    |   SGD    |      1e-3          |  48.5   |    26         |
|   6   |     16     |   5    |   SGD    |       1e-3         |  70.6   |    21       |
|   7   |     16     |   10   |   SGD    |       1e-3         |  48   |    21.5         |
|   8   |     16     |   21   |   SGD    |       1e-3         | 489.7  |    2.5       |
|   9   |     16     |   0    |   AdamW    |      1e-3          |  7.2   |    35.5         |
|   10   |     16     |   5    |   AdamW    |       1e-3         |  11.4   |    36       |
|   11   |     16     |   10   |   AdamW   |       1e-3         |  11   |    32.5         |
|  12   |     16     |   21   |   AdamW    |       1e-3         | 375  |    3        |

This table highlights the significant impact of the optimizer and the number of frozen layers on YOLOv8’s object counting performance when using a fixed learning rate of 1e-3 and batch size of 16. Among the tested optimizers, AdamW consistently outperforms others, with Model 9 (Freeze = 0) achieving the lowest MSE (7.2) and Model 10 (Freeze = 5) reaching the highest accuracy (36%). While Adam also delivers competitive results, particularly with Freeze = 5 (Model 2), its overall performance is slightly inferior to AdamW in both accuracy and error. In contrast, SGD shows the weakest performance across all freeze levels, likely due to its less adaptive update rule, leading to higher error rates and lower accuracy. Furthermore, increasing the number of frozen layers universally results in a sharp decline in performance, particularly at Freeze = 21, where all models exhibit high MSE and near-zero accuracy. These results reaffirm that full or partial fine-tuning is essential for effective transfer learning and that choosing the right optimizer—such as AdamW—can substantially enhance both precision and generalization in object counting tasks.

#  IMG SIZE EXPERIMENTS

In [ ]:
createModelTrainAndTest(optimizer='Adam',learningRate=1e-3,batchSize = 16, freezeOrNot= False, freezeValue= 0,epochVal= 25,imgSize = 416)

In [ ]:
createModelTrainAndTest(optimizer= 'Adam',learningRate= 1e-3,batchSize = 16, freezeOrNot= True , freezeValue= 5 , epochVal= 25,imgSize = 416)

In [ ]:
createModelTrainAndTest(optimizer= 'Adam',learningRate= 1e-3,batchSize = 16, freezeOrNot= True , freezeValue= 10 , epochVal= 25,imgSize = 416)

In [ ]:
createModelTrainAndTest(optimizer= 'Adam',learningRate= 1e-3,batchSize = 16, freezeOrNot= True , freezeValue= 21 , epochVal= 25,imgSize = 416)

In [ ]:
createModelTrainAndTest(optimizer= 'Adam',learningRate= 1e-3,batchSize = 16, freezeOrNot= False , freezeValue= 0 , epochVal= 25,imgSize = 960)

In [ ]:
createModelTrainAndTest(optimizer= 'Adam',learningRate= 1e-3,batchSize = 16, freezeOrNot= True , freezeValue= 5 , epochVal= 25,imgSize = 960)

In [ ]:
createModelTrainAndTest(optimizer= 'Adam',learningRate= 1e-3,batchSize = 16, freezeOrNot= True , freezeValue= 10 , epochVal= 25,imgSize = 960)

In [ ]:
createModelTrainAndTest(optimizer= 'Adam',learningRate= 1e-3,batchSize = 16, freezeOrNot= True , freezeValue= 21 , epochVal= 25,imgSize = 960)

| Model | Batch Size | Freeze | Optimizer | Learning Rate | Image Size |  MSE  | Accuracy (%) |
|-------|------------|--------|-----------|----------------|-------------|-------|---------------|
|   1   |     16     |   0    |   Adam    |      1e-3       |     640      |  8.36   |    31.5         |
|   2   |     16     |   5    |   Adam    |      1e-3       |     640      |  10.35   |    36       |
|   3   |     16     |   10   |   Adam    |      1e-3       |     640      |  11.76   |    33         |
|   4   |     16     |   21   |   Adam    |      1e-3       |     640      | 207  |  5|
|   5   |     16     |   0    |   Adam    |      1e-3       |     416      |  54.5   |    25        |
|   6   |     16     |   5    |   Adam    |      1e-3       |     416      | 87  |    20.5        |
|   7   |     16     |   10   |   Adam    |      1e-3       |     416      | 69.5  |    20.5        |
|   8   |     16     |   21   |   Adam    |      1e-3       |     416      |  666   |        1.5     |
|   9   |     16     |   0    |   Adam    |      1e-3       |     960      |   3.8  |       37     |
|   10   |     16     |   5    |   Adam    |      1e-3       |     960      | 5  |     34.5       |
|   11   |     16     |   10   |   Adam    |      1e-3       |     960      |  4 |    33.5        |
|   12   |     16     |   21   |   Adam    |      1e-3       |     960      |  126   |    7.5        |


This table clearly demonstrates the substantial impact of input image resolution on the performance of YOLOv8 models for object counting. When the image size is increased to 960×960, the models consistently outperform their counterparts trained on lower resolutions, achieving the lowest MSE (as low as 3.8) and highest accuracy (up to 37%). In contrast, models trained with 416×416 inputs perform the worst, struggling to extract fine-grained spatial details essential for accurate object detection, particularly when some layers are frozen. The intermediate 640×640 setting provides a balanced performance, but still falls short compared to 960 in both accuracy and error. Additionally, across all resolutions, freezing a large number of layers (especially Freeze = 21) consistently degrades performance, underscoring the importance of fine-tuning deeper layers when adapting YOLOv8 to new domains. These findings suggest that using higher-resolution inputs, combined with minimal freezing, is the most effective strategy for achieving accurate car counting in drone imagery.

# BEST MODEL TRY

In [ ]:
createModelTrainAndTest(optimizer='AdamW', learningRate= 1e-3, batchSize = 16, freezeOrNot= False , freezeValue= 0 , epochVal=50 ,imgSize= 960)

Among all the experiments conducted, three models stood out as top performers under different evaluation criteria. The model trained with Adam optimizer, a learning rate of 1e-3, no frozen layers, 50 epochs, and image size of 960×960 achieved the highest exact match accuracy of 42% with a competitive MSE of 4.2. This configuration proved most effective in terms of predicting the exact number of cars per image. On the other hand, the model using AdamW optimizer with the same learning rate, image size, and freeze setting yielded the lowest MSE of 3.8 and a strong accuracy of 37%, making it the most precise in terms of minimizing counting errors. Finally, another model—trained under similar conditions but evaluated separately—achieved 36% accuracy and an MSE of 4.415, which, while not the absolute best, still reflected a highly effective configuration. These results collectively highlight that using high-resolution inputs (960×960), no freezing, and either Adam or AdamW with a learning rate of 1e-3 forms the foundation of the best-performing models in this task, with slight trade-offs between absolute accuracy and error minimization depending on the optimizer used.

# Alternative Methods for Object Counting

Aside from detection-based approaches like YOLO, several alternative methods have been developed for object counting tasks. One such method is density map estimation, where a CNN generates a pixel-wise density map whose integral corresponds to the estimated count. This technique is particularly effective in crowded or low-resolution scenes where bounding boxes become unreliable. Another approach is regression-based counting, where the model directly predicts the object count as a scalar output without any spatial localization. While simple and computationally efficient, this method often lacks interpretability and performs poorly in scenes with high variability. A third method is semantic segmentation followed by connected components analysis, where the model first segments object regions and then counts them based on pixel groupings. This offers high spatial precision but relies heavily on the segmentation quality. Compared to YOLO-based detection, these alternative methods trade off spatial precision for robustness under occlusion, dense scenes, or limited annotation, and their effectiveness largely depends on the nature and complexity of the dataset.